# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyanButt1013/FlyRank_ML-Track_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Retrieve HF token safely
HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# 2. Inspect available column names in fact table
cols = [c[0] for c in con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')").fetchall()]

imp_col = 'gsc_impressions' if 'gsc_impressions' in cols else ('impressions' if 'impressions' in cols else cols[2])
click_col = 'gsc_clicks' if 'gsc_clicks' in cols else ('clicks' if 'clicks' in cols else cols[3])
pos_col = 'gsc_position' if 'gsc_position' in cols else ('gsc_avg_position' if 'gsc_avg_position' in cols else ('position' if 'position' in cols else cols[4]))

print(f"Detected columns -> Impressions: '{imp_col}', Clicks: '{click_col}', Position: '{pos_col}'")

# 3. SQL Query using NULLIF for safe division in DuckDB
query = f"""
SELECT
    c.content_hash_id,
    c.client_hash_id,
    c.word_count,
    c.content_updated_date,
    SUM(f.{imp_col}) AS impressions_90d,
    SUM(f.{click_col}) AS clicks_90d,
    AVG(f.{pos_col}) AS avg_position,
    SUM(f.{click_col}) / NULLIF(SUM(f.{imp_col}), 0) AS ctr
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet') f
  ON c.content_hash_id = f.content_hash_id
GROUP BY 1, 2, 3, 4
HAVING SUM(f.{imp_col}) >= 50
"""

df = con.sql(query).df()

# Compute content age in days relative to dataset snapshot
df['content_updated_date'] = pd.to_datetime(df['content_updated_date'])
snapshot_date = pd.to_datetime('2026-06-30')
df['days_since_update'] = (snapshot_date - df['content_updated_date']).dt.days

print(f"✅ Successfully loaded {len(df):,} valid content rows into DataFrame 'df'!")

Detected columns -> Impressions: 'gsc_impressions', Clicks: 'gsc_clicks', Position: 'gsc_avg_position'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully loaded 120,681 valid content rows into DataFrame 'df'!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# 1. My Rule and Its Reason Codes

### Plain-Words Rule Definition
Identify high-visibility pages (at least 100 impressions over 90 days) that are at risk of traffic decay due to content staleness ($\ge 180$ days since update), poor SERP snippet performance (Striking Distance position 1–20 with CTR $< 0.5\%$), or shallow word counts ($< 1,000$ words).

### Baseline Score Formula
$$\text{Baseline Score} = \min\left(100, \left(0.50 \times \text{Freshness Risk} + 0.30 \times \text{Demand Score} + 0.20 \times \text{Position Opportunity}\right) \times 100\right)$$

### Output Reason Codes
* `stale_visible_decay_risk`: Content age $\ge 180$ days with high 90-day impressions ($\ge 250$).
* `low_ctr_striking_distance`: Average position between 1 and 20 with impressions $\ge 250$ and CTR $< 0.5\%$.
* `thin_visible_content`: Word count $< 1,000$ words with impressions $\ge 250$.
* `low_volume_monitor`: Threshold criteria met, but with marginal volume ($100 \le \text{impressions} < 250$).
* `healthy_no_action`: Page performing within normal bounds; no refresh needed.

In [7]:
# 1. Print rule specification summary
print("--- BASELINE RULE SPECIFICATION ---")
print("• Core Logic: Flag high-volume pages with staleness risk, low SERP CTR, or thin word counts.")
print("• Score Formula: min(100, (0.50 * Freshness_Risk + 0.30 * Demand_Score + 0.20 * Position_Opportunity) * 100)")
print("• Action Threshold: baseline_action_score >= 50.0 -> REFRESH_CONTENT")
print("• Reason Codes: stale_visible_decay_risk, low_ctr_striking_distance, thin_visible_content, low_volume_monitor")

--- BASELINE RULE SPECIFICATION ---
• Core Logic: Flag high-volume pages with staleness risk, low SERP CTR, or thin word counts.
• Score Formula: min(100, (0.50 * Freshness_Risk + 0.30 * Demand_Score + 0.20 * Position_Opportunity) * 100)
• Action Threshold: baseline_action_score >= 50.0 -> REFRESH_CONTENT
• Reason Codes: stale_visible_decay_risk, low_ctr_striking_distance, thin_visible_content, low_volume_monitor


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
import os
import numpy as np
import pandas as pd

# 1. Ensure target output directories exist
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

# Fill missing word counts with 0 to ensure valid boolean evaluation
df['word_count_clean'] = df['word_count'].fillna(0)

# 2. Compute normalized component features
max_log_imp = np.log1p(df['impressions_90d']).max()
df['freshness_risk'] = np.clip(df['days_since_update'] / 365.0, 0, 1.5)
df['demand_score'] = np.log1p(df['impressions_90d']) / max_log_imp
df['position_opportunity'] = np.where(
    (df['avg_position'] > 0) & (df['avg_position'] <= 20),
    (20 - df['avg_position']) / 20.0,
    0.0
)

# 3. Calculate composite baseline score (0 to 100)
df['baseline_action_score'] = np.clip(
    (0.50 * df['freshness_risk'] + 0.30 * df['demand_score'] + 0.20 * df['position_opportunity']) * 100,
    0, 100
).round(2)

# 4. Map primary reason codes ensuring strict boolean arrays
conditions = [
    ((df['days_since_update'] >= 180) & (df['impressions_90d'] >= 250)).values,
    ((df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.005) & (df['impressions_90d'] >= 250)).values,
    ((df['word_count_clean'] < 1000) & (df['impressions_90d'] >= 250)).values,
    ((df['impressions_90d'] >= 100) & (df['impressions_90d'] < 250)).values
]

choices = [
    'stale_visible_decay_risk',
    'low_ctr_striking_distance',
    'thin_visible_content',
    'low_volume_monitor'
]

df['reason_code'] = np.select(conditions, choices, default='healthy_no_action')

# 5. Assign actionable label
df['action_label'] = np.where(
    df['reason_code'] != 'healthy_no_action',
    'REFRESH_CONTENT',
    'NO_ACTION'
)

# 6. Sort ranked queue descending by baseline score
ranked_queue = df.sort_values(by='baseline_action_score', ascending=False).reset_index(drop=True)

# 7. Write output CSV deliverable
output_cols = [
    'content_hash_id', 'client_hash_id', 'baseline_action_score',
    'reason_code', 'action_label', 'days_since_update', 'impressions_90d', 'avg_position'
]

ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
ranked_queue[output_cols].to_csv('../outputs/baseline_action_score.csv', index=False)

print(f"✅ Baseline queue exported successfully with {len(ranked_queue):,} rows!")
print(f"📁 Export path: work/outputs/baseline_action_score.csv")
print(f"📊 Total Actionable Recommendations: {(ranked_queue['action_label'] == 'REFRESH_CONTENT').sum():,}")

✅ Baseline queue exported successfully with 120,681 rows!
📁 Export path: work/outputs/baseline_action_score.csv
📊 Total Actionable Recommendations: 69,356


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# Select top 20 candidates from ranked queue
top_20 = ranked_queue.head(20).copy()

print("--- TOP 20 RANKED CANDIDATES AUDIT ---")
for idx, row in top_20.iterrows():
    print(f"Rank {idx+1:02d} | ID: {row['content_hash_id'][:8]}... | Score: {row['baseline_action_score']:5.1f} | "
          f"Reason: {row['reason_code']:25s} | Stale (Days): {int(row['days_since_update']):4d} | Imp: {int(row['impressions_90d']):6d}")

--- TOP 20 RANKED CANDIDATES AUDIT ---
Rank 01 | ID: content_... | Score:  73.3 | Reason: stale_visible_decay_risk  | Stale (Days):  355 | Imp:    338
Rank 02 | ID: content_... | Score:  70.2 | Reason: low_volume_monitor        | Stale (Days):  325 | Imp:    235
Rank 03 | ID: content_... | Score:  65.7 | Reason: low_volume_monitor        | Stale (Days):  322 | Imp:    223
Rank 04 | ID: content_... | Score:  65.0 | Reason: healthy_no_action         | Stale (Days):  326 | Imp:     78
Rank 05 | ID: content_... | Score:  63.8 | Reason: stale_visible_decay_risk  | Stale (Days):  214 | Imp:   8806
Rank 06 | ID: content_... | Score:  62.9 | Reason: low_volume_monitor        | Stale (Days):  323 | Imp:    137
Rank 07 | ID: content_... | Score:  62.8 | Reason: stale_visible_decay_risk  | Stale (Days):  215 | Imp:   5639
Rank 08 | ID: content_... | Score:  61.5 | Reason: stale_visible_decay_risk  | Stale (Days):  215 | Imp:   3267
Rank 09 | ID: content_... | Score:  61.3 | Reason: stale_visible_

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# 1. Assert CSV deliverable exists
assert os.path.exists("work/outputs/baseline_action_score.csv"), "CSV output file is missing!"

# 2. Check for forbidden leakage variables
forbidden_columns = ['health_score', 'priority_score', 'action_type', 'is_declining_label', 'future_traffic']
assert not any(col in df.columns for col in forbidden_columns), "Leakage error: Forbidden target/product flags found!"

print("✅ Leakage Audit Passed: No forbidden product decision flags or future windows used.")
print("✅ Output File Verified: work/outputs/baseline_action_score.csv written.")
print("🎉 Notebook execution successfully completed!")

✅ Leakage Audit Passed: No forbidden product decision flags or future windows used.
✅ Output File Verified: work/outputs/baseline_action_score.csv written.
🎉 Notebook execution successfully completed!


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.